## Back-Propagation
Implementar el algoritmo de retropropagación para un perceptrón multicapa de forma que se pueda elegir libremente la cantidad de capas de la red y de neuronas en cada capa. 

#### Algoritmo de back-propagation

```
1. Inicialización aleatoria
2. Propagación hacia adelante
3. Propagación hacia atras
4. Adaptación de los pesos
5. Iteración: vuelve a 2 hasta convergencia o finalización
```

In [14]:
import numpy as np
import pandas as pd
import random as random
import copy


class Capa:
    w : np.ndarray
    y: np.ndarray
    delta: np.ndarray

    def __init__(self, w_i, y_i, delta_i):
        self.w = w_i
        self.y = y_i
        self.delta = delta_i

    def mostrar(self):
        print(f"Pesos: {self.w}")
        print(f"Salidas: {self.y}")
        print(f"Deltas: {self.delta}\n")


def sigm(x):
    return (2/(1+np.exp(-x))) - 1

entrada_usuario = [2,1]

tabla = pd.read_csv('../../Data/gtp-1/XOR_trn.csv', header=None).to_numpy()
x0 = -np.ones(len(tabla))
entradas = np.c_[x0, tabla[:,:-1]] # indice -1 := ultima columna ( Acceso a indices con : es [) )
yd =  tabla[:, -1]

print(entradas.shape)


w = np.random.rand(entrada_usuario[0], len(entradas[0])) - 0.5 # dimension: 0 == columnas, dimension: 1 == filas
y_init = np.zeros(entrada_usuario[0])
delta = np.zeros(entrada_usuario[0])
cap = Capa(w,y_init,delta)
vect_capas = [copy.deepcopy(cap)]

# Iniciar red (aleatorio):
for i in range(1,len(entrada_usuario)):
    w = np.random.rand(entrada_usuario[i], entrada_usuario[i-1]+1) - 0.5
    y_init = np.zeros(entrada_usuario[i])
    delta = np.zeros(entrada_usuario[i])
    cap = Capa(w,y_init,delta)
    vect_capas.append(copy.deepcopy(cap))


# Visualizar red inicial:
print(f"Cantidad de capas: {len(vect_capas)}\n")
print(f"Red neuronal: \n")
i = 1
for capa in vect_capas:
    print(f"Capa: {i}")
    capa.mostrar()
    i += 1


(2000, 3)
Cantidad de capas: 2

Red neuronal: 

Capa: 1
Pesos: [[-0.11786857  0.25800489 -0.43975936]
 [ 0.46503211  0.20809293  0.27359018]]
Salidas: [0. 0.]
Deltas: [0. 0.]

Capa: 2
Pesos: [[-0.15103938  0.4774991   0.35441885]]
Salidas: [0.]
Deltas: [0.]



#### Entrenar la red

In [15]:
# Iterar sobre la red:

epoca = 1
epoca_max = 500
mu = 0.1

# n -> ejemplo actual
# i -> la capa
# j -> la neurona
while epoca < epoca_max: 
    # SOLUCIÓN: Iterar sobre len(entradas) (las filas), no len(entradas[0]) (las columnas)
    for n in range(len(entradas)):

        # paso hacia adelante
        for i in range(len(vect_capas)):
            for j in range(len(vect_capas[i].y)):
                if i==0:
                    z = np.dot(entradas[n,:],vect_capas[i].w[j,:])
                else: 
                    ent = np.r_[-1,vect_capas[i-1].y]
                    z = np.dot(ent,vect_capas[i].w[j,:])
                vect_capas[i].y[j] = sigm(z)

        # propagacion hacia atras
        # SOLUCIÓN: Empezar en len(vect_capas)-1 para no tener un IndexError
        for i in range(len(vect_capas)-1,-1,-1):
            for j in range(len(vect_capas[i].y)):
                # SOLUCIÓN: Restar 1 al if porque los índices empiezan en 0
                if i==len(vect_capas)-1:
                    # SOLUCIÓN: Usar yd[n] (el deseado del ejemplo actual) en vez de yd[j]
                    vect_capas[i].delta[j] = (1/2) * (yd[n] - vect_capas[i].y[j]) * (1 + vect_capas[i].y[j]) * (1 - vect_capas[i].y[j])
                else:
                    # SOLUCIÓN: np.dot contra el array de deltas completo de la capa siguiente (quitamos el [j])
                    vect_capas[i].delta[j] = (1/2) * np.dot(vect_capas[i+1].delta, vect_capas[i+1].w[:,j+1])  * (1 + vect_capas[i].y[j]) * (1 - vect_capas[i].y[j])

        #actualizar los pesos
        for i in range(len(vect_capas)):
            for j in range(len(vect_capas[i].y)):
                # SOLUCIÓN: Usar len de los pesos de la neurona actual
                for m in range(len(vect_capas[i].w[j])):
                    if i==0:
                        # SOLUCIÓN: Usar += en lugar de -=
                        vect_capas[i].w[j,m] += mu*vect_capas[i].delta[j]*entradas[n,m]
                    else:
                        # SOLUCIÓN: Hay que reconstruir la entrada con el -1 para poder usar el índice 'm' sin salir de rango, y usar +=
                        ent = np.r_[-1, vect_capas[i-1].y]
                        vect_capas[i].w[j,m] += mu*vect_capas[i].delta[j]*ent[m]
                        
    # Verificación:
    acierto = 0
    for n in range(len(entradas)):
        for capa in range(len(vect_capas)):
            for neuron in range(len(vect_capas[capa].y)):
                if capa==0:
                    z = np.dot(entradas[n,:],vect_capas[capa].w[neuron,:])
                else:                
                    ent = np.r_[-1,vect_capas[capa-1].y]
                    z = np.dot(ent,vect_capas[capa].w[neuron,:])

                vect_capas[capa].y[neuron] = sigm(z)

        salida_red = vect_capas[-1].y[0]
        if ((salida_red >= 0.85 and yd[n] == 1) or (salida_red <= -0.85 and yd[n] == -1)):
            acierto += 1

    tasa_acierto = acierto/len(entradas)
    print(f"Fin entrenamiento. Epoca: {epoca}, Tasa de acierto: {tasa_acierto * 100: .2f}\n")

    if tasa_acierto > 0.90:
        print(f"Convergencia. Tasa de aciertos: {tasa_acierto}, Epoca: {epoca}\n")
        break
    else:
        epoca += 1

print(f"Red neuronal final: \n")
i = 1
for cap in vect_capas:
    print(f"Capa {i}:")
    cap.mostrar()
    i += 1



Fin entrenamiento. Epoca: 1, Tasa de acierto:  0.00

Fin entrenamiento. Epoca: 2, Tasa de acierto:  0.00

Fin entrenamiento. Epoca: 3, Tasa de acierto:  100.00

Convergencia. Tasa de aciertos: 1.0, Epoca: 3

Red neuronal final: 

Capa 1:
Pesos: [[ 2.17006401  2.54353407 -2.53760175]
 [ 2.69589721 -2.92492288  2.9344687 ]]
Salidas: [-0.99853882  0.91610519]
Deltas: [6.21167120e-05 3.36010341e-03]

Capa 2:
Pesos: [[-3.27425167  3.96923199  3.89882305]]
Salidas: [0.89395425]
Deltas: [0.01071535]



### Test: XOR

Testear el modelo con datos de XOR_test.

In [16]:
tabla_tst = pd.read_csv('../../Data/gtp-1/XOR_tst.csv', header=None).to_numpy()

x0 = -np.ones(len(tabla_tst))
entradas = np.c_[x0, tabla_tst[:,:-1]] # indice -1 := ultima columna ( Acceso a indices con : es [) )
yd =  tabla_tst[:, -1]

print(f"Cantidad de capas: {len(vect_capas)}\n")
print(f"Red neuronal: \n")
i = 1
for cap in vect_capas:
    print(f"Capa {i}:")
    cap.mostrar()
    i += 1

acierto = 0
for n in range(len(entradas)):
    for capa in range(len(vect_capas)):
        for neuron in range(len(vect_capas[capa].y)):
            if capa==0:
                z = np.dot(entradas[n,:],vect_capas[capa].w[neuron,:])
            else:                
                ent = np.r_[-1,vect_capas[capa-1].y]
                z = np.dot(ent,vect_capas[capa].w[neuron,:])

            vect_capas[capa].y[neuron] = sigm(z)

    salida_red = vect_capas[-1].y[0]
    if ((salida_red >= 0.85 and yd[n] == 1) or (salida_red <= -0.85 and yd[n] == -1)):
        acierto += 1

tasa_acierto = acierto/len(entradas)
print(f"Test: Tasa de acierto: {tasa_acierto * 100: .2f}%")


Cantidad de capas: 2

Red neuronal: 

Capa 1:
Pesos: [[ 2.17006401  2.54353407 -2.53760175]
 [ 2.69589721 -2.92492288  2.9344687 ]]
Salidas: [-0.99853882  0.91610519]
Deltas: [6.21167120e-05 3.36010341e-03]

Capa 2:
Pesos: [[-3.27425167  3.96923199  3.89882305]]
Salidas: [0.89395425]
Deltas: [0.01071535]

Test: Tasa de acierto:  100.00%
